## Do Marine Observations Occur in Distinct Ocean Chemical Conditions?

In [ ]:
# Let's start with our imports!

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from scipy.spatial import cKDTree

# Datasets Used

Dataset 1: Dissolved Inorganic Carbon & Total Alkalinity

- 
- 
- 

Dataset 2: OBIS Seamap Dataset Points

- 
- 
- 

In [ ]:
chem_df = pd.read_csv("CALCOFI_DIC_20250122.csv") # dataset 1
seamap_p_df = pd.read_csv("obis_seamap_dataset_507_points.csv") # dataset 2

# Data Cleaning


In [ ]:
# Cleaning dataset 1
chem_df = chem_df.iloc[1:].reset_index(drop=True)
chem_df = chem_df.replace(-999, np.nan)
chem_df['datetime'] = pd.to_datetime(
    chem_df['Year_UTC'].astype(int).astype(str) + '-' +
    chem_df['Month_UTC'].astype(int).astype(str) + '-' +
    chem_df['Day_UTC'].astype(int).astype(str) + ' ' +
    chem_df['Time_UTC'].astype(str)
)
chem_df = chem_df.rename(columns={
    'Year_UTC': 'year',
    'Month_UTC': 'month'
})
chem_df = chem_df.drop(columns=['EXPOCODE', 'Ship_Name', 'Station_ID'])
chem_cols = ['DIC', 'TA', 'CTDTEMP_ITS90', 'Salinity_PSS78', 'Latitude', 'Longitude']
for col in chem_cols:
    chem_df[col] = pd.to_numeric(chem_df[col], errors='coerce')

chem_df = chem_df.dropna(subset=chem_cols)

# Cleaning dataset 2
seamap_p_df = seamap_p_df.drop(columns=['dataset_id', 'row_id', 'series_id', 'itis_tsn', 'lprecision', 'tprecision', 'notes', 'last_mod', 'timezone', 'provider', 'platform', 'oceano'])
seamap_p_df['date_time'] = pd.to_datetime(seamap_p_df['date_time'], errors='coerce')
seamap_p_df = seamap_p_df.dropna(subset=['date_time'])
seamap_p_df['year'] = seamap_p_df['date_time'].dt.year
seamap_p_df['month'] = seamap_p_df['date_time'].dt.month
seamap_p_df = seamap_p_df.dropna(subset=['latitude', 'longitude'])

# Visualizations

### OBIS Sampling Intensity Heatmap

In [ ]:
plt.figure(figsize=(8,6))

plt.hexbin(
    seamap_p_df['longitude'],
    seamap_p_df['latitude'],
    gridsize=50,
    cmap='Reds',
    mincnt=1
)

plt.colorbar(label="Observation density")
plt.title("OBIS Sampling Intensity Heatmap")
plt.show()

Biological observations are unevenly distributed, with strong clustering in certain regions—important for interpreting downstream patterns.

### Ocean Chemistry Regimes

In [ ]:
plt.figure(figsize=(8,6))

plt.hexbin(
    chem_df['Longitude'],
    chem_df['Latitude'],
    C=chem_df['DIC'],
    reduce_C_function=np.mean,
    gridsize=50,
    cmap='viridis'
)

plt.colorbar(label="Mean DIC")
plt.title("Ocean Chemistry Regimes (DIC)")
plt.show()

DIC shows clear spatial structure, indicating distinct chemical regimes across the study area.

### Overlay: Observations on DIC Map

In [ ]:
plt.figure(figsize=(8,6))

plt.hexbin(
    chem_df['Longitude'],
    chem_df['Latitude'],
    C=chem_df['DIC'],
    reduce_C_function=np.mean,
    gridsize=50,
    cmap='viridis',
    alpha=0.6
)

plt.scatter(
    seamap_p_df['longitude'],
    seamap_p_df['latitude'],
    color='red',
    s=3,
    alpha=0.3
)

plt.colorbar(label="DIC")
plt.title("Marine Observations vs Ocean Chemistry Regimes")
plt.show()

Observations tend to cluster within specific chemical regions, suggesting a potential relationship between marine presence and DIC conditions.

### Quantifying the relationship between observations and chemistry

In [ ]:
chem_coords = np.column_stack((chem_df['Latitude'], chem_df['Longitude']))
tree = cKDTree(chem_coords)
bio_coords = np.column_stack((seamap_p_df['latitude'], seamap_p_df['longitude']))
dist, idx = tree.query(bio_coords, k=1)
seamap_p_df['matched_DIC'] = chem_df.iloc[idx]['DIC'].values

plt.figure(figsize=(8,5))

sns.kdeplot(chem_df['DIC'], label='All Ocean DIC', fill=True, alpha=0.4)
sns.kdeplot(seamap_p_df['matched_DIC'], label='At Observation Locations', fill=True, alpha=0.4)

plt.xlabel("DIC")
plt.ylabel("Density")
plt.title("Do Marine Observations Occur in Specific DIC Conditions?")
plt.legend()
plt.show()

DIC distribution: overall vs observation locations

The DIC distribution at observation locations is shifted toward higher values compared to the background ocean. This suggests that marine observations are not randomly distributed, but instead occur preferentially under specific chemical conditions.